<font color='red'><b>**WARNING**</b></font> <br/>
어떠한 사유로도 임의로 복사, 촬영, 녹음, 복제, 보관, 전송하거나 허가 받지 않은 저장매체를 이용한 보관, 제3자에게 누설, 공개 또는 사용하는 등의 무단 사용 및 불법 배포 시 법적 조치를 받을 수 있습니다. <br/>

<div style="text-align: right; color: #7f8c8d; font-size: 0.9em; margin-top: 20px;">
📝 Author: 박사홍 (Sahong Pak)</br>
📧 Contact: sahong.pak@gmail.com</br>
📌 Version: v2.0</br>
📅 Last Updated: 2026-03-23</br>
</div>

</br>

# 학습 내용
>이번 장에서는 <strong>LLM as a Judge(LLM 기반 자동 평가)</strong>에 대해 학습합니다.</br></br>
>LLM을 평가자로 활용하여 합성 데이터의 품질을 자동으로 학습해봅시다.

</br>

<div style="text-align:center">

</div>

In [1]:
# TODO 0: LLM 호출에 필요한 라이브러리를 불러오고, LLM 객체를 생성해봅시다.

from dotenv import load_dotenv
from langchain_upstage import ChatUpstage

load_dotenv()

llm = ChatUpstage(model="solar-pro3", temperature=0)

/Users/sahong.pak/Documents/AI/.venv/lib/python3.14/site-packages/langchain_core/_api/deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1
/Users/sahong.pak/Documents/AI/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


</br>

# LLM as a Judge
> <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">LLM을 평가자(Judge)로 활용</mark>하여 텍스트의 품질을 자동으로 점수화하는 기법입니다.
> 인간 평가의 비용과 시간을 절약하면서 일정 수준의 품질 관리가 가능합니다.

> 이전 챕터에서 합성 데이터 1,000개를 생성했다면, 그 품질을 어떻게 검증할까요? 사람이 하나씩 읽는 방법은 현실적으로 불가능합니다.</br></br>
> 크라우드소싱 플랫폼에서 전문 평가자를 고용하면 샘플당 수백 원에서 수천 원의 비용이 발생하지만, LLM 평가는 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">동일 비용의 1% 수준</mark>으로 가능합니다. 또한 사람은 피로, 기분, 해석 차이에 따라 평가 기준이 흔들리는 반면, LLM은 동일한 프롬프트를 사용하면 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">일관된 기준</mark>으로 수천 건을 평가합니다.</br></br>
> 데이터가 10만 개로 늘어나도 LLM 평가는 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">비례하여 확장</mark>되므로 대규모 평가에 적합합니다.</br></br>
> 합성 데이터 품질 검증 외에도 RAG 시스템의 답변 품질 평가, 챗봇 응답 평가, A/B 테스트 등 다양한 용도로 활용되며, 평가 루브릭(Rubric, 채점 기준표)을 명확히 정의하고 JSON으로 결과를 파싱하면 자동화된 품질 관리 파이프라인을 구축할 수 있습니다.

<table style="width:100%">
  <thead>
    <tr>
      <th style="text-align:center">장점</th>
      <th>한계</th>
    </tr>
  </thead>
  <tbody>
    <tr><td style="text-align:center">빠른 대규모 평가</td><td>인간 판단과 완전 일치하지 않음</td></tr>
    <tr><td style="text-align:center">일관된 기준 적용</td><td>평가 프롬프트에 민감</td></tr>
    <tr><td style="text-align:center">비용 효율적</td><td>자기 편향(Self-bias) 가능</td></tr>
  </tbody>
</table>

</br>

## 평가 프롬프트 설계
> 평가 기준을 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">명확하고 구체적으로 정의</mark>하는 것이 핵심입니다.

In [2]:
# TODO 1: 구체성/자연스러움/유용성 3가지 기준(각 1-5점)으로 리뷰 품질을 평가하고 JSON으로 답변하도록 요청하는 평가 프롬프트 템플릿을 작성해봅시다.

eval_prompt = """다음 리뷰의 품질을 1-5점으로 평가하세요.

평가 기준:
1. 구체성 (1-5): 제품의 구체적인 특징을 언급하는가?
2. 자연스러움 (1-5): 실제 고객이 쓴 것처럼 자연스러운가?
3. 유용성 (1-5): 다른 구매자에게 도움이 되는 정보가 있는가?

리뷰: {review_text}

JSON 형식으로 답변:
{{"specificity": N, "naturalness": N, "usefulness": N, "total": N, "reason": "..."}}"""

</br>

## 점수화 기준 (Rubric)
> 각 점수 구간의 의미를 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">명시적으로 정의</mark>합니다.

<table style="width:100%">
  <thead>
    <tr>
      <th style="text-align:center">점수</th>
      <th>기준</th>
    </tr>
  </thead>
  <tbody>
    <tr><td style="text-align:center">5</td><td>매우 우수 — 구체적이고 자연스러우며 유용</td></tr>
    <tr><td style="text-align:center">4</td><td>우수 — 대부분 기준 충족</td></tr>
    <tr><td style="text-align:center">3</td><td>보통 — 일부 기준만 충족</td></tr>
    <tr><td style="text-align:center">2</td><td>미흡 — 대부분 기준 미달</td></tr>
    <tr><td style="text-align:center">1</td><td>부적합 — 품질 기준에 크게 미달</td></tr>
  </tbody>
</table>

</br>

## 자동 평가 파이프라인

In [3]:
# TODO 2: 리뷰 텍스트를 평가 프롬프트로 LLM에 전달하여 점수를 반환하는 함수를 정의해봅시다.

def evaluate_review(review_text, llm):
    """LLM으로 리뷰 품질 평가"""
    prompt = eval_prompt.format(review_text=review_text)
    response = llm.invoke(prompt)

    import json
    scores = json.loads(response.content)
    return scores

In [4]:
# TODO 3: 3개의 샘플 리뷰를 반복문으로 배치 평가한 결과를 출력해봅시다.

reviews = [
    "이 제품 좋아요.",
    "배터리가 8시간 지속되고 무게가 200g이라 휴대성이 뛰어납니다. 음질도 선명해요.",
    "가격 대비 괜찮은데 통화 품질은 조금 아쉽습니다. 노이즈 캔슬링은 만족스러워요."
]

results = []
for review in reviews:
    score = evaluate_review(review, llm)
    results.append(score)
    print(f"점수: {score['total']}/15 | 이유: {score['reason'][:40]}...")

점수: 2/15 | 이유: 리뷰가 '이 제품 좋아요'라는 매우 일반적인 표현만 포함하고 있어 구체성...
점수: 4/15 | 이유: 리뷰는 배터리 지속 시간(8시간), 무게(200g), 음질(선명함) 등 ...
점수: 3/15 | 이유: 리뷰는 가격 대비 만족도와 노이즈 캔슬링에 대한 구체적인 언급을 포함하고...


</br>

## 필터링 (Quality Filtering)
> 평가 점수를 기준으로 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">저품질 데이터를 제거</mark>합니다.

In [5]:
# TODO 4: 품질 기준 점수를 설정하고, 평가 결과에서 기준 이상인 항목만 필터링하여 통과율을 출력해봅시다.

threshold = 3.5
filtered = [r for r in results if r["total"] / 3 >= threshold]

print(f"원본: {len(results)}개 → 필터링 후: {len(filtered)}개")
print(f"통과율: {len(filtered)/len(results)*100:.1f}%")

원본: 3개 → 필터링 후: 0개
통과율: 0.0%


💡평가 신뢰도 높이기
> 동일 데이터를 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">여러 번 평가</mark>하거나 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">다른 모델로 교차 평가</mark>하면 신뢰도가 높아집니다.

💡평가 모델은 어떤 것을 사용할까?
> 생성 모델보다 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">동급 이상의 모델</mark>을 평가자로 사용하는 것이 권장됩니다.</br>
> GPT-4로 생성한 데이터를 GPT-4로 평가하면 자기 편향이 생길 수 있습니다.